In [1]:
import utils
import sbmodel
import pycantus
import pycantus.data as data

In [2]:
corpus = data.load_dataset('cantuscorpus_v1.0')

Loading chants and sources...
Data loaded!


In [3]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 888010
Number of sources in CantusCorpus v1.0: 2278


In [4]:
# Drop doxology
doxo_filter = pycantus.filtration.Filter('doxo_filter')
doxo_filter.add_value_exclude('cantus_id', '909000')
corpus.apply_filter(doxo_filter)

In [81]:
MIN_CHANTS_PER_SOURCE = 600

In [7]:
# Drop fragments => sources with less than 100 chants
corpus.drop_small_sources_data(min_chants=MIN_CHANTS_PER_SOURCE)

In [8]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 812806
Number of sources in CantusCorpus v1.0: 363


In [9]:
sigla_dict = {source.srclink: source.siglum for source in corpus.sources}

### Networks

In [10]:
import importlib
importlib.reload(utils)

<module 'utils' from '/home/watticka/network_analysis/project/ChantNets/experiments/utils.py'>

In [55]:
MIN_CID_PER_FEAST = 50
MIN_FEAST_CHANTS_PER_SOURCE = 10

In [56]:
graph = utils.construct_bipart_source_feast_reducted_graph(corpus, min_cid_per_feast=MIN_CID_PER_FEAST, 
                                                           per_source_threshold=MIN_FEAST_CHANTS_PER_SOURCE)

Constructing bipartite graph between sources and feasts...
Number of source nodes: 363
Number of feast nodes: 719
Number of source-feast edges: 17638


In [57]:
utils.save_graph(graph, f"nets/source_feast_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_bi_graph_min_{MIN_CHANTS_PER_SOURCE}.gt")

### SBM

In [58]:
N_INIT = 20

In [59]:
model = sbmodel.SBModel()

In [60]:
model.load_graph(f"nets/source_feast_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_bi_graph_min_{MIN_CHANTS_PER_SOURCE}.gt")

Loaded graph with 1082 vertices, 17638 edges


In [61]:
model.fit_sbm(n_init=N_INIT)

Fitting SBM to graph with 1082 vertices and 17638 edges...
Fitting SBM (init 1/20)...
[1/20] entropy = 41789.92
Fitting SBM (init 2/20)...
[2/20] entropy = 41694.33
Fitting SBM (init 3/20)...
[3/20] entropy = 41688.49
Fitting SBM (init 4/20)...
[4/20] entropy = 41691.36
Fitting SBM (init 5/20)...
[5/20] entropy = 41760.45
Fitting SBM (init 6/20)...
[6/20] entropy = 41734.55
Fitting SBM (init 7/20)...
[7/20] entropy = 41674.23
Fitting SBM (init 8/20)...
[8/20] entropy = 41764.46
Fitting SBM (init 9/20)...
[9/20] entropy = 41826.97
Fitting SBM (init 10/20)...
[10/20] entropy = 41758.91
Fitting SBM (init 11/20)...
[11/20] entropy = 41677.11
Fitting SBM (init 12/20)...
[12/20] entropy = 41739.53
Fitting SBM (init 13/20)...
[13/20] entropy = 41702.36
Fitting SBM (init 14/20)...
[14/20] entropy = 41752.75
Fitting SBM (init 15/20)...
[15/20] entropy = 41757.06
Fitting SBM (init 16/20)...
[16/20] entropy = 41764.63
Fitting SBM (init 17/20)...
[17/20] entropy = 41808.34
Fitting SBM (init 18/20)

In [62]:
model.save_states(f"models/dc_sbm_best_states_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.pickle")

In [63]:
model.fit_nested_sbm(n_init=N_INIT)

Fitting nested SBM to graph with 1082 vertices and 17638 edges...
Fitting nested SBM (init 1/20)...
[1/20] entropy = 40548.29
Fitting nested SBM (init 2/20)...
[2/20] entropy = 40527.14
Fitting nested SBM (init 3/20)...
[3/20] entropy = 40546.93
Fitting nested SBM (init 4/20)...
[4/20] entropy = 40564.36
Fitting nested SBM (init 5/20)...
[5/20] entropy = 40531.09
Fitting nested SBM (init 6/20)...
[6/20] entropy = 40655.28
Fitting nested SBM (init 7/20)...
[7/20] entropy = 40498.42
Fitting nested SBM (init 8/20)...
[8/20] entropy = 40537.75
Fitting nested SBM (init 9/20)...
[9/20] entropy = 40585.76
Fitting nested SBM (init 10/20)...
[10/20] entropy = 40637.48
Fitting nested SBM (init 11/20)...
[11/20] entropy = 40539.96
Fitting nested SBM (init 12/20)...
[12/20] entropy = 40520.28
Fitting nested SBM (init 13/20)...
[13/20] entropy = 40543.91
Fitting nested SBM (init 14/20)...
[14/20] entropy = 40574.78
Fitting nested SBM (init 15/20)...
[15/20] entropy = 40552.02
Fitting nested SBM (in

In [64]:
model.save_states(f"models/dc_nested_sbm_best_states_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.pickle")

In [65]:
model.fit_sbm_weighted(n_init=N_INIT, weight_label='weight')

Fitting weighted SBM to graph with 1082 vertices and 17638 edges...
Fitting weighted SBM (init 1/20)...
[1/20] entropy = 86667.92
Fitting weighted SBM (init 2/20)...
[2/20] entropy = 86650.34
Fitting weighted SBM (init 3/20)...
[3/20] entropy = 86696.09
Fitting weighted SBM (init 4/20)...
[4/20] entropy = 86784.72
Fitting weighted SBM (init 5/20)...
[5/20] entropy = 86637.14
Fitting weighted SBM (init 6/20)...
[6/20] entropy = 86619.33
Fitting weighted SBM (init 7/20)...
[7/20] entropy = 86714.30
Fitting weighted SBM (init 8/20)...
[8/20] entropy = 86705.94
Fitting weighted SBM (init 9/20)...
[9/20] entropy = 86725.19
Fitting weighted SBM (init 10/20)...
[10/20] entropy = 86707.01
Fitting weighted SBM (init 11/20)...
[11/20] entropy = 86674.58
Fitting weighted SBM (init 12/20)...
[12/20] entropy = 86637.75
Fitting weighted SBM (init 13/20)...
[13/20] entropy = 86794.16
Fitting weighted SBM (init 14/20)...
[14/20] entropy = 86645.96
Fitting weighted SBM (init 15/20)...
[15/20] entropy =

In [66]:
model.save_states(f"models/weighted_dc_sbm_best_states_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.pickle")

In [67]:
model.fit_nested_sbm_weighted(n_init=N_INIT, weight_label='weight')

Fitting weighted nested SBM to graph with 1082 vertices and 17638 edges...
Fitting weighted nested SBM (init 1/20)...
[1/20] entropy = 86685.79
Fitting weighted nested SBM (init 2/20)...
[2/20] entropy = 86698.05
Fitting weighted nested SBM (init 3/20)...
[3/20] entropy = 86827.22
Fitting weighted nested SBM (init 4/20)...
[4/20] entropy = 86790.82
Fitting weighted nested SBM (init 5/20)...
[5/20] entropy = 86714.04
Fitting weighted nested SBM (init 6/20)...
[6/20] entropy = 86647.19
Fitting weighted nested SBM (init 7/20)...
[7/20] entropy = 86714.22
Fitting weighted nested SBM (init 8/20)...
[8/20] entropy = 86686.91
Fitting weighted nested SBM (init 9/20)...
[9/20] entropy = 86723.56
Fitting weighted nested SBM (init 10/20)...
[10/20] entropy = 86732.76
Fitting weighted nested SBM (init 11/20)...
[11/20] entropy = 86691.99
Fitting weighted nested SBM (init 12/20)...
[12/20] entropy = 86782.74
Fitting weighted nested SBM (init 13/20)...
[13/20] entropy = 86676.09
Fitting weighted nes

In [68]:
model.save_states(f"models/weighted_nested_dc_sbm_best_states_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.pickle")

### Infering

In [69]:
import pickle
importlib.reload(sbmodel)
importlib.reload(utils)

<module 'utils' from '/home/watticka/network_analysis/project/ChantNets/experiments/utils.py'>

In [70]:
model = sbmodel.SBModel()

In [71]:
model.load_states(f'models/weighted_nested_dc_sbm_best_states_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.pickle')

In [72]:
print(model.best_states.keys())

dict_keys(['DC_SBM', 'Nested_DC_SBM', 'Weighted_DC_SBM', 'Weighted_Nested_DC_SBM'])


#### DC-SBM

In [73]:
best_state = model.best_states['DC_SBM']  

In [74]:
print("Best DC SBM state:")
print(best_state)
index_partitions, sigla_partitions, feasts_partitions = utils.get_partitions_from_state(best_state, sigla_dict)
print('Number of partitions:', len(index_partitions))
print('Number of sigla partitions:', len(sigla_partitions))
print('Number of feast partitions:', len(feasts_partitions))

Best DC SBM state:
<BlockState object with 1082 blocks (22 nonempty), degree-corrected, for graph <Graph object, undirected, with 1082 vertices and 17638 edges, 2 internal vertex properties, 2 internal edge properties, at 0x74ea86ce45e0>, at 0x74ea86ce72e0>
Number of partitions: 22
Number of sigla partitions: 10
Number of feast partitions: 12


In [75]:
with open(f'visual/dc_sbm_feast_partitions{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}min_{MIN_CHANTS_PER_SOURCE}.txt', 'w') as f:
    for partition, feasts in feasts_partitions.items():
        print(sorted(feasts), file=f)
        print(file=f)

In [78]:
importlib.reload(utils)
print(utils.detect_sanctorale_in_partitions(feasts_partitions))

Partition 875: 29/72 : {'Dec': 9, 'Jan': 12, 'Feb': 3, 'Mar': 2, 'Apr': 1, 'May': 1, 'Sep': 1}
Partition 446: 4/17 : {'Dec': 3, 'Jan': 1}
Partition 402: 148/240 : {'Jul': 24, 'Aug': 21, 'Jun': 13, 'Feb': 6, 'May': 11, 'Nov': 15, 'Sep': 20, 'Jan': 6, 'Oct': 16, 'Apr': 6, 'Dec': 8, 'Mar': 2}
Partition 593: 13/18 : {'Nov': 3, 'Jun': 4, 'Aug': 2, 'Sep': 2, 'May': 1, 'Oct': 1}
Partition 557: 6/27 : {'Jun': 1, 'Sep': 1, 'Oct': 1, 'Nov': 2, 'Jul': 1}
Partition 561: 17/38 : {'Jul': 2, 'Jun': 1, 'Aug': 6, 'Apr': 1, 'Oct': 2, 'May': 2, 'Sep': 1, 'Nov': 2}
Partition 378: 1/46 : {'Dec': 1}
Partition 806: 2/14 : {'Dec': 1, 'Aug': 1}
Partition 684: 10/15 : {'Jan': 3, 'Dec': 1, 'Feb': 4, 'Mar': 2}
Partition 907: 37/131 : {'Jun': 1, 'Dec': 6, 'Jan': 4, 'Jul': 4, 'Aug': 8, 'Sep': 6, 'Nov': 3, 'Feb': 1, 'Oct': 2, 'May': 1, 'Apr': 1}
Partition 839: 2/53 : {'Oct': 1, 'Dec': 1}
Partition 1029: 12/48 : {'Apr': 1, 'Feb': 1, 'Dec': 1, 'Jun': 3, 'Jul': 2, 'Nov': 1, 'Sep': 1, 'May': 1, 'Aug': 1}
None


#### DC-Weighted-SBM

In [79]:
best_state = model.best_states['Weighted_DC_SBM']

In [80]:
print("Best Weighted DC SBM state:")
print(best_state)
index_partitions, sigla_partitions, feasts_partitions = utils.get_partitions_from_state(best_state, sigla_dict)
print('Number of partitions:', len(index_partitions))
print('Number of sigla partitions:', len(sigla_partitions))
print('Number of feast partitions:', len(feasts_partitions))

Best Weighted DC SBM state:
<BlockState object with 1082 blocks (22 nonempty), degree-corrected, with 1 edge covariate, for graph <Graph object, undirected, with 1082 vertices and 17638 edges, 2 internal vertex properties, 2 internal edge properties, at 0x74ea86ce45e0>, at 0x74ea84acf820>
Number of partitions: 22
Number of sigla partitions: 10
Number of feast partitions: 12


In [42]:
with open(f'visual/weighted_dc_sbm_feast_partitions{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}__threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.txt', 'w') as f:
    for partition, feasts in feasts_partitions.items():
        print(sorted(feasts), file=f)
        print(file=f)

In [76]:
importlib.reload(utils)
source_vs_feast_df = utils.get_sigla_vs_feast_partitions(corpus, sigla_partitions, feasts_partitions)

In [77]:
source_vs_feast_df.to_csv(f'visual/weighted_dc_sbm_source_vs_feast_partitions{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_min_{MIN_CHANTS_PER_SOURCE}.csv', index=False)

#### DC-Nested-SBM

In [47]:
best_state = model.best_states['Nested_DC_SBM']

In [48]:
importlib.reload(utils)
print("Best Nested DC SBM state:")
print(best_state)
index_partitions, sigla_partitions, feasts_partitions = utils.get_nested_partitions_from_state(best_state, sigla_dict)

Best Nested DC SBM state:
<NestedBlockState object, with base <BlockState object with 1082 blocks (60 nonempty), degree-corrected, for graph <Graph object, undirected, with 1082 vertices and 39601 edges, 2 internal vertex properties, 2 internal edge properties, at 0x74ea85998160>, at 0x74ea75aa5ab0>, and 12 levels of sizes [(1082, 60), (60, 18), (18, 5), (5, 3), (3, 2), (2, 2), (2, 2), (2, 2), (2, 2), (2, 2), (2, 2), (2, 2)] at 0x74ea86819de0>


In [49]:
print('Feast partitions:')
for level in feasts_partitions:
    print(f'Level {level}:')
    print(len(feasts_partitions[level]))
    for partition, feasts in feasts_partitions[level].items():
        print(f'Partition {partition}: {len(feasts)} {sorted(feasts)}')
    print()

Feast partitions:
Level 0:
26
Partition 470: 13 ['Blasii', 'Conceptio Mariae', 'Conversio Pauli', 'De BMV post Purif.', 'Hilarii', 'Mauri', 'Nicolai', 'Octava Agnetis', 'Octava Andreae', 'Scholasticae', 'Thomae Apost.', 'Thomae Cant.', 'Vedasti']
Partition 402: 14 ['Dom. 1 Adventus', 'Dom. 1 Quadragesimae', 'Dom. 2 Adventus', 'Dom. 2 Quadragesimae', 'Dom. 3 Adventus', 'Dom. 3 Quadragesimae', 'Dom. 4 Adventus', 'Dom. 4 Quadragesimae', 'Dom. Quinquagesimae', 'Dom. Septuagesimae', 'Dom. Sexagesimae', 'Nativitas Domini,8', 'Vigilia Epiphaniae', 'Vigilia Nat. Domini']
Partition 397: 44 ['Barbarae', 'Dom. 4 p. Epiph.', 'Dom. 5 p. Epiph.', 'Dom. p. Oct. Nat.', 'Fer. 2 Hebd. 2 p.Ep.', 'Fer. 2 Hebd. 3 Adv.', 'Fer. 2 p. Epiphaniam', 'Fer. 3 Hebd. 2 p.Ep.', 'Fer. 3 Hebd. 3 Adv.', 'Fer. 3 p. Epiphaniam', 'Fer. 4 Hebd. 1 Adv.', 'Fer. 4 Hebd. 2 p.Ep.', 'Fer. 4 Hebd. 3 Adv.', 'Fer. 4 p. Epiphaniam', 'Fer. 5 Hebd. 2 p.Ep.', 'Fer. 5 Hebd. 3 Adv.', 'Fer. 5 p. Epiphaniam', 'Fer. 6 Hebd. 1 Adv.', 'Fer. 6 

In [50]:
for level in sigla_partitions:
    print(f'Level {level}:')
    print(len(sigla_partitions[level]))
    for partition, sigla in sigla_partitions[level].items():
        print(f'Partition {partition}: {len(sigla)} : {sorted(sigla)}')
    print()

Level 0:
34
Partition 26: 20 : ['A-Gu 29 (olim 38/8 f.)', 'A-KN CCl 1010', 'A-KN CCl 1011', 'A-KN CCl 1015', 'A-KN CCl 1017', 'A-Wda C-11', 'CH-SGs 390', 'CZ-Pu XIV.B.13', 'D-MZb A', 'D-Mbs Clm 4303', 'DK-Kk Gl. Kgl. S. 3449 8o [02] II', 'F-Pn Lat. 9449', 'GB-Lbl Add MS 30847', 'H-Bu lat. 118', 'HR-Hf Cod. C Nr. 4', 'HR-Hf Cod. D Nr. 5', 'I-BV V 19', 'I-MC 542', 'PL-Kkar Ms.2 (rkp. Perg. 14)', 'PL-Kkar Ms.5 (rkp. Perg. 13)']
Partition 117: 6 : ['A-Gu 30 (olim 38/9 f.)', 'CZ-Bu R 387', 'CZ-Pu VI E 13', 'CZ-Pu XII C 3', 'CZ-Pu XIII C 4', 'CZ-Pu XIV C 20']
Partition 247: 3 : ['A-LIb 290 (olim 183; olim Gamma p 19)', 'A-Wn Cod. 1890', 'CZ-Pu VI G 11']
Partition 0: 12 : ['A-SF XI 480', 'CH-E 611', 'CH-SGs 388', 'D-B Mus. 40047', 'D-KA Aug. LX', 'D-KNd 215', 'D-Sl HB.I.55', 'E-Tc 44.1', 'F-Pn Latin 12044', 'GB-Ob MS. Laud Misc. 284', 'I-PCd cod. 65', 'I-Rv C.5']
Partition 95: 6 : ['B-TOb 63 (olim V)', 'B-TOb 64 (olim IV)', 'CDN-Hsmu M2149.L4 1554', 'CZ-Pu VI.E.4c', 'D-KNd 1161', 'I-VCd CLXX'

In [84]:
importlib.reload(utils)
source_vs_feast_df = utils.get_sigla_vs_feast_partitions(corpus, sigla_partitions[1], feasts_partitions[1])

In [86]:
source_vs_feast_df.head(10)

,source_partition,n_of_sources,f_p_46,f_p_49,f_p_21,f_p_30,f_p_50,f_p_38
0,8,86,0.020,0.312,0.229,0.221,0.040,0.0
1,51,54,0.036,0.004,0.002,0.197,0.029,0.0
2,1,32,0.068,0.390,0.267,0.465,0.095,0.0
3,34,26,0.002,0.143,0.112,0.059,0.022,0.0
4,28,71,0.122,0.511,0.037,0.497,0.002,0.0
5,25,8,0.016,0.098,0.066,0.109,0.019,0.0
6,3,56,0.108,0.347,0.028,0.328,0.005,0.0
7,55,30,0.013,0.217,0.013,0.132,0.008,0.0


In [ ]:
source_vs_feast_df.to_csv(f'visual/nested_dc_sbm_source_vs_feast_partitions{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}_levels_1_1.csv', index=False)

In [51]:
save_path = f"partitions/nested_dc_sbm_partitions_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}.csv"
df_partitions = utils.save_nested_partitions(sigla_partitions, save_path)
df_partitions.head()

Saved nested partitions to partitions/nested_dc_sbm_partitions_20_f_reducted_50_threshold_5_min_600.csv


,siglum,level_0,level_1,level_2,level_3,level_4,level_5,level_6,level_7,level_8,...,level2_new,level3_new,level4_new,level5_new,level6_new,level7_new,level8_new,level9_new,level10_new,level11_new
0,A-Gu 29 (olim 38/8 f.),26,42,1,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CZ-Pu XIV.B.13,26,42,1,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,A-KN CCl 1010,26,42,1,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,A-KN CCl 1011,26,42,1,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,A-KN CCl 1015,26,42,1,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [52]:
for col in df_partitions.columns:
    if col.startswith("level") and col.endswith("_new"):
        print(col, df_partitions[col].nunique())

level0_new 34
level1_new 7
level2_new 3
level3_new 1
level4_new 1
level5_new 1
level6_new 1
level7_new 1
level8_new 1
level9_new 1
level10_new 1
level11_new 1


In [53]:
COLS_4 = ["siglum", "level0_new", "level1_new", "level2_new", "level3_new"]

In [54]:
importlib.reload(utils)
utils.get_dendro_json(save_path, columns=COLS_4, output_path=f"visual/nested_dc_sbm_dendro_{N_INIT}_f_reducted_{MIN_CID_PER_FEAST}_threshold_{MIN_FEAST_CHANTS_PER_SOURCE}_min_{MIN_CHANTS_PER_SOURCE}_cols4.json")